# This script computes statistics that are specific to germany, comparing the months Jan-Apr of the original reporting period (2012-2025) to the current year (2026)
Note, that most of this code is identical to that in [03_compute_drought_days.ipynb](03_compute_drought_days.ipynb). Creating a diff between the two files should be the most convenient way to do fact checking for the first cells here.

In [1]:
import numpy as np
import re
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import rasterio
import glob
import os
import utils

from tqdm.notebook import tqdm

tqdm.pandas()

In [2]:
cdi_files = sorted(
    glob.glob(
        os.path.join(
            utils.raw_data_dir, "combined_drought_indicator_2012-2025", "**", "*.tif"
        )
    )
)
cdi_folders = sorted(
    glob.glob(
        os.path.join(utils.raw_data_dir, "combined_drought_indicator_2012-2025", "*")
    )
)

# like before, toggle this variable to switch between LAU and NUTS-3 level of analysis
LAU_level = False
if LAU_level:
    raster_input_dir = "cdi_raster_coverage_lau"
else:
    raster_input_dir = "cdi_raster_coverage_nuts3"

In [3]:
# load only the country-file for germany into a single DataFrame
df = gpd.read_file(os.path.join(utils.intermediate_data_dir, raster_input_dir, "DE.geojson"))
lau_crs = df.crs 

In [4]:
# load the first cdi file to extract metadata
with rasterio.open(cdi_files[0]) as src:
    geotiff = src.read(1)
    cdi_dims = src.read(1).shape

    assert src.crs == lau_crs

    # compute pixel indices in the raster for each pair of x/y coordinates
    min_coords = np.array(
        [src.index(x, y) for x, y in df[["xmin", "ymin"]].values.astype(float)]
    )
    max_coords = np.array(
        [src.index(x, y) for x, y in df[["xmax", "ymax"]].values.astype(float)]
    )


# enter pixel indieces into the dataframe for later reference
df["row_start"] = min_coords[:, 0]
df["col_start"] = min_coords[:, 1]
df["row_end"] = max_coords[:, 0]
df["col_end"] = max_coords[:, 1]

In [5]:
# minor type cleanup
for col in ["col_start", "col_end", "row_start", "row_end"]:
    df[col] = df[col].astype(int)
df["cell_coverage"] = df["cell_coverage"].apply(lambda x: np.array(eval(x)))


def calculate_cell_weights(cell_coverage):
    # by default, exactextract returns the fraction of each grid cell that is being covered by the polygon.
    # We however desire the inverse: How much of the Polygon's area lies in a given grid cell
    # The latter allows us to weigh grid values to determine the average value inside the LAU
    total_coverage = cell_coverage.sum(axis=None)
    return cell_coverage / total_coverage


df["cell_weights"] = df["cell_coverage"].apply(calculate_cell_weights)

In [6]:
# compute two 3D-matrices containing the total number of "warning" and "alert" days per pixel in the CDI raster
warning_days = np.zeros(((2027 - 2012), cdi_dims[0], cdi_dims[1]))
alert_days = np.zeros(((2027 - 2012), cdi_dims[0], cdi_dims[1]))

for i, year in enumerate(range(2012, 2027)):
    print("processing year: ", year)

    year_folder = cdi_folders[i]
    year_files = sorted(glob.glob(os.path.join(year_folder, "*.tif")))
    yearly_data = np.zeros(
        shape=(12, cdi_dims[0], cdi_dims[1]), dtype=np.uint8
    )

    for file_idx, path in enumerate(year_files):
        # NOTE: this is the important bit: in this script we only compare the first four months of all years
        if file_idx > 11:
            break
        with rasterio.open(path) as src:
            yearly_data[file_idx] = src.read(1)

    # value 2 corresponds to CDI level "Warning", value 3 corresponds to level "Alert".
    # See page 3 https://drought.emergency.copernicus.eu/data/factsheets/factsheet_combinedDroughtIndicator_v4.pdf
    cur_year_warning = yearly_data == 2
    warning_days[i] = cur_year_warning.sum(axis=0) * 10

    cur_year_alert = yearly_data == 3
    alert_days[i] = cur_year_alert.sum(axis=0) * 10

processing year:  2012
processing year:  2013
processing year:  2014
processing year:  2015
processing year:  2016
processing year:  2017
processing year:  2018
processing year:  2019
processing year:  2020
processing year:  2021
processing year:  2022
processing year:  2023
processing year:  2024
processing year:  2025
processing year:  2026


In [7]:
def calculate_yearly_stats(row):
    """generates three sets of arrays for each row in the dataframe: "warning days", "alert days" and their sum: "drought days". For this, it looks at the subgrid of the CDI stats computed in the previous cell.

    Args:
        row (pd.Series): a row in the dataframe, representing either a LAU or NUTS-3

    Returns:
        pd.Series: the same row, including three new columns
    """
    cell_weights = row["cell_weights"]
    width, height = cell_weights.shape

    # NOTE: @factcheck: I messed this up previously. I believe it to be right like this, but all of this dimension swapping can be rather confusing. Please double check.
    x0, x1 = row["row_start"] - width, row["row_start"]
    y0, y1 = row["col_start"], row["col_start"] + height

    # take the subset of the 3D-arrays that overlaps with the shape of this region
    # then, multiply each value by the amount of the total area it comprises
    # finaly, summ over axes 2 and 3 to get a 1D-array representing the "warning" and "alert" days per year.
    warning_days_per_year = (
        (warning_days[:, x0:x1, y0:y1] * cell_weights).sum(axis=(1, 2)).round(2)
    )
    alert_days_per_year = (
        (alert_days[:, x0:x1, y0:y1] * cell_weights).sum(axis=(1, 2)).round(2)
    )

    row["warning_days"] = warning_days_per_year.tolist()
    row["alert_days"] = alert_days_per_year.tolist()
    row["drought_days"] = (
        (warning_days_per_year + alert_days_per_year).round(2).tolist()
    )
    return row


print("calculating yearly stats...")
df = df.progress_apply(calculate_yearly_stats, axis=1)

calculating yearly stats...


  0%|          | 0/401 [00:00<?, ?it/s]

In [8]:
# compute median value, maximum value and the year that the maximum value occurred for each of the three metrices
# NOTE: all secondary metrics are only computed up until 2025, so we can compare the 2026 values to previous median and max values.
for value in ["warning", "alert", "drought"]:
    df[f"median_{value}_days"] = df[f"{value}_days"].apply(
        lambda x: np.median(x[:-1]).round(2)
    )
    df[f"max_{value}_days"] = df[f"{value}_days"].apply(lambda x: np.max(x[:-1]).round(2))
    df[f"max_{value}_days_year"] = df[f"{value}_days"].apply(
        lambda x: 2012 + np.argmax(x[:-1])
    )
    df[f"2026_{value}_days"] = df[f"{value}_days"].apply(lambda x: x[-1])
    df[f"2026_{value}_days"] = df[f"{value}_days"].apply(lambda x: x[-1])

In [9]:
# drop columns that are no longer needed
if LAU_level:
    out_data = gpd.GeoDataFrame(
        df[
            [
                "CNTR_CODE",
                "LAU_NAME",
                "geometry",
                "warning_days",
                "median_warning_days",
                "max_warning_days",
                "max_warning_days_year",
                "alert_days",
                "median_alert_days",
                "max_alert_days",
                "max_alert_days_year",
                "drought_days",
                "median_drought_days",
                "max_drought_days",
                "max_drought_days_year",
            ]
        ]
    )
else:
    out_data = gpd.GeoDataFrame(
        df[
            [
                "NUTS_ID",
                "CNTR_CODE",
                "NAME_LATN",
                "NUTS_NAME",
                "geometry",
                "warning_days",
                "median_warning_days",
                "max_warning_days",
                "max_warning_days_year",
                "alert_days",
                "median_alert_days",
                "max_alert_days",
                "max_alert_days_year",
                "drought_days",
                "median_drought_days",
                "max_drought_days",
                "max_drought_days_year",
            ]
        ]
    )

out_data = out_data.to_crs(epsg=4326)

In [10]:
df = df.to_crs(epsg=3035) # convert to a metric CRS to compute surface area
df["area_km2"] = round(df.geometry.area / 1e6, 2)

In [11]:
df[df["max_drought_days"] < df["2026_drought_days"]]
# in this cell we select all rows where the value for 2026 is higher than the maximum of all previously recorded years.

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,cell_coverage,xmin,...,2026_warning_days,median_alert_days,max_alert_days,max_alert_days_year,2026_alert_days,median_drought_days,max_drought_days,max_drought_days_year,2026_drought_days,area_km2
24,DE923,3,DE,Hameln-Pyrmont,Hameln-Pyrmont,4,2,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.002, 0.289, ...",9.125000,...,95.15,0.00,28.57,2012,0.48,5.70,80.27,2019,95.63,797.83
164,DE732,3,DE,Fulda,Fulda,4,2,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0321, 0...",9.416667,...,108.05,0.02,14.18,2012,0.00,18.51,101.31,2019,108.05,1381.51
248,DE916,3,DE,Goslar,Goslar,3,2,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.00...",10.041667,...,98.01,0.39,20.57,2012,0.00,31.58,91.76,2019,98.01,968.65
339,DEA57,3,DE,Hochsauerlandkreis,Hochsauerlandkreis,2,2,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",7.833333,...,62.06,0.45,22.38,2012,0.21,29.02,60.67,2014,62.27,1959.26


# --> Hameln-Pyrmont, Fulda, Goslar und der Hochsauerlandkreis haben in den ersten vier Monaten des Jahres so viele DürreWARNtage erlebt, wie in keinem anderen ersten Jahresdrittel im Aufzeichnungszeitraum.

In [17]:
df[df["max_alert_days"] < df["2026_alert_days"]]
# in this cell we select all rows where the number of alert days for 2026 is higher than the maximum of all previously recorded years.

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,cell_coverage,xmin,...,2026_warning_days,median_alert_days,max_alert_days,max_alert_days_year,2026_alert_days,median_drought_days,max_drought_days,max_drought_days_year,2026_drought_days,area_km2
105,DE21G,3,DE,Mühldorf a. Inn,Mühldorf a. Inn,4,3,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...",12.000000,...,49.04,0.0,4.53,2012,4.67,15.06,102.14,2020,53.71,805.05
223,DEE0D,3,DE,Stendal,Stendal,4,3,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.6764e-06, 0....",11.375000,...,0.42,0.0,2.09,2022,2.11,0.68,119.49,2019,2.53,2436.22
349,DEE01,3,DE,"Dessau-Roßlau, Kreisfreie Stadt","Dessau-Roßlau, Kreisfreie Stadt",4,2,3,"[[0.0, 0.0, 0.0004, 0.1486, 0.1893, 0.0, 0.0],...",12.083333,...,67.10,0.0,3.83,2021,4.44,3.55,120.00,2019,71.54,246.03


# --> Mühldorf a. Inn, Stendal und Dessau-Roßlau haben in den ersten vier Monaten des Jahres so viele DürreALARMtage erlebt, wie in keinem anderen ersten Jahresdrittel im Aufzeichnungszeitraum.

In [18]:
mean_drought_days_baseline = (df["median_drought_days"]*df["area_km2"]).sum() / df["area_km2"].sum()
mean_drought_days_2026 = (df["2026_drought_days"]*df["area_km2"]).sum()  / df["area_km2"].sum()
f"Mean warning days between 2012-2025: {mean_drought_days_baseline:.2f}. 2026 was {mean_drought_days_2026:.2f}, marking an increase of {100*((mean_drought_days_2026/mean_drought_days_baseline)-1):.2f} per cent."

'Mean warning days between 2012-2025: 14.47. 2026 was 27.33, marking an increase of 88.87 per cent.'

In [19]:
mean_alert_days_baseline = (df["median_alert_days"]*df["area_km2"]).sum() / df["area_km2"].sum()
mean_alert_days_2026 = (df["2026_alert_days"]*df["area_km2"]).sum()  / df["area_km2"].sum()
f"Mean alert days between 2012-2025: {mean_alert_days_baseline:.2f}. 2026 was {mean_alert_days_2026:.2f}, marking an increase of {100*((mean_alert_days_2026/mean_alert_days_baseline)-1):.2f} per cent."

'Mean alert days between 2012-2025: 0.08. 2026 was 0.51, marking an increase of 567.96 per cent.'

# --> das erste Drittel 2026 (Januar-April) war überdurchschnittlich trocken. Im Vergleich zu den vorherigen Jahren seit 2012 hat Deutschland in diesem Frühjahr über sechsmal so viele Tage erlebt, an denen die Dürre örtlich so schwer war, dass sie das Pflanzenwachstum beeinträchtigte.

In [20]:
# NUTS-2 code for Schleswig holstein is DEF0 https://de.wikipedia.org/wiki/NUTS:DE
df_schleswig_holstein = df[df["NUTS_ID"].apply(lambda x: x.startswith("DEF0"))]

In [21]:
mean_alert_days_baseline = (df_schleswig_holstein["median_warning_days"]*df_schleswig_holstein["area_km2"]).sum() / df_schleswig_holstein["area_km2"].sum()
mean_alert_days_2026 = (df_schleswig_holstein["2026_warning_days"]*df_schleswig_holstein["area_km2"]).sum()  / df_schleswig_holstein["area_km2"].sum()
f"Mean alert days between 2012-2025: {mean_alert_days_baseline:.2f}. 2026 was {mean_alert_days_2026:.2f}, marking an increase of {100*((mean_alert_days_2026/mean_alert_days_baseline)-1):.2f} per cent."

'Mean alert days between 2012-2025: 3.26. 2026 was 6.38, marking an increase of 96.00 per cent.'

# --> Schleswig Holstein erlebte ein ungewöhnlich trockenes erstes Jahresdrittel: hier wurden etwa zweimal so viele Dürrewarntage verzeichnet, wie in einem durchschnittsjahr zwischen 2012 und 2025.